# Lenguajes y Autómatas II (SCD-1016)
## Sesión 23 · Etiquetas y Saltos Incondicionales (Goto)

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Cómo se le dice al procesador que continúe en otra parte del programa sin perder el orden lógico?
>
> **Objetivo de la sesión:** Generar etiquetas únicas y las dos formas de un salto incondicional: la marca de destino y el GOTO que la usa.

**Entregable:** tus funciones nueva_etiqueta() y emitir_flujo() en artesano_sesion23_c.c, el harness en verde y la bitácora en auditorias/auditoria_sesion23.md

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)


## 1. ¿Qué vas a construir?

Recibes un contador de etiquetas y, según el modo, debes escribir GOTO Lx o Lx:. La función no imprime: escribe en dst y, cuando corresponde, avanza el contador.

Al final de la sesión tendrás: **Dos funciones en C: una que emite etiquetas L0, L1... y otra que escribe un GOTO o la marca de destino, probadas con seis casos.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 23](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion23.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion23.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion23.h"

void nueva_etiqueta(char *dst, int cap, int *contador);
void emitir_flujo(char *dst, int cap, const char *etiqueta, int es_salto);
```

- nueva_etiqueta() usa el valor actual y después lo incrementa: la primera etiqueta es L0.
- emitir_flujo() con es_salto distinto de cero escribe GOTO; con cero escribe la marca con dos puntos.
- Una etiqueta NULL produce cadena vacía; nunca escribas fuera de cap.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion23_c.c` junto con `artesano_sesion23_c.c`
> (o `ia_sesion23_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `nueva_etiqueta(L0)` | `L0` |
| 2 | `nueva_etiqueta(L5)` | `L5` |
| 3 | `emitir_flujo(GOTO L2)` | `GOTO L2` |
| 4 | `emitir_flujo(marca L2)` | `L2:` |
| 5 | `emitir_flujo(NULL)` | `(cadena vacía)` |
| 6 | `nueva_etiqueta(L10)` | `L10` |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 23 - Etiquetas y saltos incondicionales ===
Caso                               esperado:              obtenido:              resultado
[ 1] nueva_etiqueta(L0)                 esperado: L0           obtenido: L0           OK
[ 2] nueva_etiqueta(L5)                 esperado: L5           obtenido: L5           OK
[ 3] emitir_flujo(GOTO L2)              esperado: GOTO L2      obtenido: GOTO L2      OK
[ 4] emitir_flujo(marca L2)             esperado: L2:          obtenido: L2:          OK
[ 5] emitir_flujo(NULL)                 esperado:              obtenido:              OK
[ 6] nueva_etiqueta(L10)                esperado: L10          obtenido: L10          OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion23.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] Las etiquetas generadas son únicas y consecutivas: L0, L1, L2...
- [ ] GOTO y la marca de destino usan exactamente el mismo nombre de etiqueta.
- [ ] Una etiqueta NULL no provoca fallo de segmentación y produce cadena vacía.


## Paso 0 · Entorno


In [ ]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion23_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [ ]:
%%writefile api_sesion23.h
/* api_sesion23.h - contrato de la Sesion 23
 * Etiquetas y saltos incondicionales
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION23_H
#define API_SESION23_H

void nueva_etiqueta(char *dst, int cap, int *contador);
void emitir_flujo(char *dst, int cap, const char *etiqueta, int es_salto);

#endif /* API_SESION23_H */


In [ ]:
%%writefile artesano_sesion23_c.c
/* artesano_sesion23_c.c - Fase 1: El Artesano (sin IA)
 * Etiquetas y saltos incondicionales.
 *
 * Implementa lo que declara api_sesion23.h.
 * El harness compila este archivo junto con el suyo: no escribas main().
 */
#include "api_sesion23.h"
#include <stdio.h>
#include <string.h>

void nueva_etiqueta(char *dst, int cap, int *contador) {
          (void)dst; (void)cap; (void)contador; /* TODO: escribe LK y avanza el contador */
      }
      
      void emitir_flujo(char *dst, int cap, const char *etiqueta, int es_salto) {
          (void)dst; (void)cap; (void)etiqueta; (void)es_salto; /* TODO: GOTO o marca de destino */
      }


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion23_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


In [ ]:
%%writefile ia_sesion23_c.c
/* ia_sesion23_c.c - Fase 2: El Cientifico
 *
 * Pega aqui el codigo que te devolvio la IA, con la MISMA firma
 * que declara api_sesion23.h. No puede traer main().
 */
#include "api_sesion23.h"

/* TODO: pega aqui la propuesta de la IA */


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [ ]:
%%writefile harness_sesion23_c.c
/* harness_sesion23_c.c - auditoria automatica de la Sesion 23
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion23_c.c artesano_sesion23_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion23.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_str(const char *caso, const char *esperado,
                          const char *obtenido) {
    int ok = (obtenido != NULL) && (strcmp(esperado, obtenido) == 0);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12s obtenido: %-12s %s\n",
           probadas, caso, esperado, obtenido ? obtenido : "(NULL)",
           ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 23 - Etiquetas y saltos incondicionales ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { int c = 0; char b[32] = ""; nueva_etiqueta(b, 32, &c); verificar_str("nueva_etiqueta(L0)", "L0", b); }
    { int c = 5; char b[32] = ""; nueva_etiqueta(b, 32, &c); verificar_str("nueva_etiqueta(L5)", "L5", b); }
    { char b[32] = ""; emitir_flujo(b, 32, "L2", 1); verificar_str("emitir_flujo(GOTO L2)", "GOTO L2", b); }
    { char b[32] = ""; emitir_flujo(b, 32, "L2", 0); verificar_str("emitir_flujo(marca L2)", "L2:", b); }
    { char b[32] = ""; emitir_flujo(b, 32, NULL, 1); verificar_str("emitir_flujo(NULL)", "", b); }
    { int c = 10; char b[32] = ""; nueva_etiqueta(b, 32, &c); verificar_str("nueva_etiqueta(L10)", "L10", b); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion23.md" "\n");
    return 0;
}


In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion23_c.c artesano_sesion23_c.c -o test_artesano && ./test_artesano

In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion23_c.c ia_sesion23_c.c -o test_ia && ./test_ia

## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion23.md` a `auditorias/auditoria_sesion23.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
